# Document-Level Analysis

Visualizations for the document-level dataset produced by `build_datasets.py`. Focuses on distributions of detections, coverage, heaviness, and categorical breakdowns by redaction level.

In [ ]:
# Cell 1: Install Dependencies
!pip install matplotlib seaborn

In [ ]:
# Cell 2: Imports & Configuration
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
PALETTE = "Set2"
DEFAULT_FIGSIZE = (10, 6)
WIDE_FIGSIZE = (14, 5)
HIST_KWARGS = dict(bins=30, edgecolor="black", alpha=0.7)

In [ ]:
# Cell 3: Load Data
# >>> UPDATE THIS PATH to point to your document-level CSV <<<
DATA_PATH = "../datasets/document_level.csv"

df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
print()
df.dtypes

## Distributions

Histograms are filtered to rows where the plotted value > 0 to avoid a large spike at zero.

In [ ]:
# Cell 4: Distribution of total_detections per document
data = df.loc[df["total_detections"] > 0, "total_detections"]

fig, ax = plt.subplots(figsize=DEFAULT_FIGSIZE)
ax.hist(data, **HIST_KWARGS, color=sns.color_palette(PALETTE)[0])
ax.set_title("Distribution of Total Detections per Document (>0)")
ax.set_xlabel("Total Detections")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 5: Distribution of total_pages per document
data = df.loc[df["total_pages"] > 0, "total_pages"]

fig, ax = plt.subplots(figsize=DEFAULT_FIGSIZE)
ax.hist(data, **HIST_KWARGS, color=sns.color_palette(PALETTE)[1])
ax.set_title("Distribution of Total Pages per Document (>0)")
ax.set_xlabel("Total Pages")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6: Distribution of mean_coverage_percent
data = df.loc[df["mean_coverage_percent"] > 0, "mean_coverage_percent"]

fig, ax = plt.subplots(figsize=DEFAULT_FIGSIZE)
ax.hist(data, **HIST_KWARGS, color=sns.color_palette(PALETTE)[2])
ax.set_title("Distribution of Mean Coverage Percent per Document (>0)")
ax.set_xlabel("Mean Coverage (%)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7: Distribution of mean_heaviness_score
data = df.loc[df["mean_heaviness_score"] > 0, "mean_heaviness_score"]

fig, ax = plt.subplots(figsize=DEFAULT_FIGSIZE)
ax.hist(data, **HIST_KWARGS, color=sns.color_palette(PALETTE)[3])
ax.set_title("Distribution of Mean Heaviness Score per Document (>0)")
ax.set_xlabel("Mean Heaviness Score")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8: Distribution of fraction_pages_with_redactions
data = df.loc[df["fraction_pages_with_redactions"] > 0, "fraction_pages_with_redactions"]

fig, ax = plt.subplots(figsize=DEFAULT_FIGSIZE)
ax.hist(data, **HIST_KWARGS, color=sns.color_palette(PALETTE)[4])
ax.set_title("Distribution of Fraction of Pages with Redactions (>0)")
ax.set_xlabel("Fraction")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## Categorical

In [ ]:
# Cell 9: Count bar chart — modal_redaction_level
level_order = ["none", "light", "moderate", "heavy"]
level_counts = df["modal_redaction_level"].value_counts().reindex(level_order).fillna(0)

fig, ax = plt.subplots(figsize=DEFAULT_FIGSIZE)
level_counts.plot.bar(ax=ax, color=sns.color_palette(PALETTE)[:len(level_order)], edgecolor="black")
ax.set_title("Modal Redaction Level Counts")
ax.set_xlabel("Redaction Level")
ax.set_ylabel("Number of Documents")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 10: Bar chart — has_any_redaction (True vs False)
redaction_counts = df["has_any_redaction"].value_counts()

fig, ax = plt.subplots(figsize=DEFAULT_FIGSIZE)
redaction_counts.plot.bar(ax=ax, color=sns.color_palette(PALETTE)[:2], edgecolor="black")
ax.set_title("Documents With vs Without Any Redaction")
ax.set_xlabel("Has Any Redaction")
ax.set_ylabel("Number of Documents")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 11: Stacked horizontal bar — classification type composition per redaction level
class_cols = ["total_inline", "total_partial_line", "total_full_line", "total_block"]
level_order = ["none", "light", "moderate", "heavy"]

grouped = df.groupby("modal_redaction_level")[class_cols].sum()
grouped = grouped.reindex(level_order).fillna(0)

fig, ax = plt.subplots(figsize=WIDE_FIGSIZE)
grouped.plot.barh(stacked=True, ax=ax, colormap="Set2", edgecolor="black")
ax.set_title("Classification Type Composition per Redaction Level")
ax.set_xlabel("Total Count")
ax.set_ylabel("Modal Redaction Level")
ax.legend(title="Classification", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## Relationships

In [ ]:
# Cell 12: Scatter — total_pages vs total_detections
fig, ax = plt.subplots(figsize=DEFAULT_FIGSIZE)
ax.scatter(df["total_pages"], df["total_detections"],
           alpha=0.5, edgecolors="black", linewidths=0.5,
           color=sns.color_palette(PALETTE)[0])
ax.set_title("Total Pages vs Total Detections")
ax.set_xlabel("Total Pages")
ax.set_ylabel("Total Detections")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 13: Violin + box plot — mean_coverage_percent by modal_redaction_level
level_order = ["none", "light", "moderate", "heavy"]
plot_df = df[df["modal_redaction_level"].isin(level_order)].copy()
plot_df["modal_redaction_level"] = pd.Categorical(
    plot_df["modal_redaction_level"], categories=level_order, ordered=True
)

fig, ax = plt.subplots(figsize=DEFAULT_FIGSIZE)
sns.violinplot(data=plot_df, x="modal_redaction_level", y="mean_coverage_percent",
               palette=PALETTE, inner=None, ax=ax)
sns.boxplot(data=plot_df, x="modal_redaction_level", y="mean_coverage_percent",
            width=0.15, boxprops=dict(facecolor="white", zorder=2),
            ax=ax, fliersize=0)
ax.set_title("Mean Coverage Percent by Modal Redaction Level")
ax.set_xlabel("Modal Redaction Level")
ax.set_ylabel("Mean Coverage (%)")
plt.tight_layout()
plt.show()